# T57-企业预警通爬取

## 项目简介
本项目是一个用于爬取企业预警通(www.qyyjt.cn)网站数据的爬虫框架，支持API和Selenium两种方式。

---
## 第一章：环境配置与依赖导入

In [ ]:
# 第一章：环境配置与依赖导入
import pandas as pd
import numpy as np
import sqlalchemy
import os
import json
import warnings
warnings.filterwarnings('ignore')

# 导入配置
from config import DB_YQ_URL, DATA_DIR, OUTPUT_DIR, DATA_MODULES
from src import QYYJTSpider, QYYJTSeleniumSpider

print("依赖导入完成！")

---
## 第二章：数据库连接与爬虫初始化

In [ ]:
# 第二章：数据库连接与爬虫初始化
# 创建数据库连接
sql_engine = sqlalchemy.create_engine(DB_YQ_URL, poolclass=sqlalchemy.pool.NullPool)
cursor = sql_engine.connect()

print("数据库连接成功！")

# 获取登录凭据
username = os.environ.get('QYYJT_USERNAME', '')
password = os.environ.get('QYYJT_PASSWORD', '')

print(f"用户名已配置: {'是' if username else '否'}")
print(f"密码已配置: {'是' if password else '否'}")

# 初始化API爬虫
spider = QYYJTSpider(username, password)
print("\nAPI爬虫初始化完成！")

---
## 第三章：用户登录与Token获取

In [ ]:
# 第三章：用户登录与Token获取
# 执行登录
if username and password:
    login_success = spider.login()
    if login_success:
        print(f"登录成功!")
        print(f"用户ID: {spider.user_id}")
        print(f"Token: {spider.token[:20]}..." if spider.token else "Token: None")
    else:
        print("登录失败，请检查用户名和密码")
else:
    print("请配置环境变量 QYYJT_USERNAME 和 QYYJT_PASSWORD")

---
## 第四章：API方式爬取数据

In [ ]:
# 第四章：API方式爬取数据
import requests
import random
import time
import hashlib

def get_default_headers():
    """获取默认请求头"""
    return {
        'authority': 'www.qyyjt.cn',
        'accept': '*/*',
        'accept-encoding': 'gzip, deflate, br',
        'accept-language': 'zh-CN,zh;q=0.9,en;q=0.8',
        'cache-control': 'no-cache',
        'client': 'pc-web;pro',
        'content-type': 'application/json;charset=UTF-8',
        'origin': 'https://www.qyyjt.cn',
        'pragma': 'no-cache',
        'sec-fetch-dest': 'empty',
        'sec-fetch-mode': 'cors',
        'sec-fetch-site': 'same-origin',
        'system': 'new',
        'terminal': 'pc-web;pro',
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'ver': '20240624'
    }

def fetch_data_via_api(url, data, headers, session):
    """通过API获取数据"""
    try:
        response = session.post(url, headers=headers, json=data, timeout=30)
        if response.status_code == 200:
            result = response.json()
            return result
    except Exception as e:
        print(f"请求失败: {e}")
    return None

# 使用爬虫获取市场化经营主体数据
if spider.token:
    print("正在获取市场化经营主体数据...")
    df_market = spider.get_market_table()
    
    if df_market is not None:
        print(f"\n获取到 {len(df_market)} 条数据")
        print(f"\n列名: {df_market.columns.tolist()}")
        display(df_market.head())
    else:
        print("获取数据失败")
        df_market = pd.DataFrame()
else:
    print("未登录，无法获取数据")
    df_market = pd.DataFrame()

---
## 第五章：Selenium方式爬取数据（备选）

In [ ]:
# 第五章：Selenium方式爬取数据（备选）
# 注意：Selenium需要安装对应的浏览器驱动

USE_SELENIUM = False  # 设置为True以使用Selenium

if USE_SELENIUM:
    print("使用Selenium方式爬取...")
    
    selenium_spider = QYYJTSeleniumSpider(username, password)
    
    try:
        # 登录
        if selenium_spider.login():
            # 获取表格数据
            df_selenium = selenium_spider.get_market_table()
            
            if df_selenium is not None:
                print(f"\n获取到 {len(df_selenium)} 条数据")
                display(df_selenium.head())
            else:
                print("获取数据失败")
        else:
            print("Selenium登录失败")
    finally:
        selenium_spider.close()
else:
    print("跳过Selenium方式（USE_SELENIUM=False）")

---
## 第六章：数据存储到数据库

In [ ]:
# 第六章：数据存储到数据库
if not df_market.empty:
    table_name = '市场化经营主体'
    
    # 添加爬取时间戳
    df_market['crawl_time'] = pd.Timestamp.now()
    
    try:
        with sql_engine.begin() as connection:
            df_market.to_sql(table_name, connection, if_exists='append', index=False)
        print(f"成功保存 {len(df_market)} 条数据到表 {table_name}")
    except Exception as e:
        print(f"保存失败: {e}")
else:
    print("没有数据需要保存")

# 保存到Excel
if not df_market.empty:
    output_file = os.path.join(OUTPUT_DIR, 'market_data.xlsx')
    spider.save_to_excel(df_market, output_file)
    print(f"数据已保存到: {output_file}")

---
## 第七章：数据验证与导出

In [ ]:
# 第七章：数据验证与导出
# 验证数据库中的数据
try:
    verify_sql = "SELECT COUNT(*) as cnt FROM 市场化经营主体"
    result = pd.read_sql(verify_sql, cursor)
    print(f"表 市场化经营主体 中共有 {result['cnt'].iloc[0]} 条数据")
    
    # 显示最新数据
    sample_sql = "SELECT * FROM 市场化经营主体 ORDER BY crawl_time DESC LIMIT 5"
    sample = pd.read_sql(sample_sql, cursor)
    print("\n最新5条数据:")
    display(sample)
except Exception as e:
    print(f"验证失败: {e}")

# 生成报告
report = {
    'crawl_time': str(pd.Timestamp.now()),
    'total_records': len(df_market) if not df_market.empty else 0,
    'columns': df_market.columns.tolist() if not df_market.empty else [],
    'status': 'success' if not df_market.empty else 'no_data'
}

report_file = os.path.join(OUTPUT_DIR, 'crawl_report.json')
with open(report_file, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(f"\n爬取报告已保存到: {report_file}")

# 关闭连接
cursor.close()
print("\n数据库连接已关闭")